
# Fraud Detection in Credit Card Transactions (End-to-End)

**Objective:** Identify fraudulent transactions using anomaly detection and a supervised model.

**Dataset:** Kaggle's Credit Card Fraud Detection dataset (`creditcard.csv`).  
Download from Kaggle and place `creditcard.csv` in the **same folder** as this notebook.

**Includes:**
- Preprocessing & class balancing
- Unsupervised anomaly detection (Isolation Forest & Local Outlier Factor)
- Supervised model (XGBoost)
- Confusion matrices & ROC curve
- Saved artifacts for deployment (Streamlit app)


In [ ]:

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ML & preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, RocCurveDisplay, precision_recall_fscore_support
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Imbalanced data handling
try:
    from imblearn.under_sampling import RandomUnderSampler
except Exception as e:
    raise RuntimeError("Please install imbalanced-learn: pip install imbalanced-learn") from e

# XGBoost
try:
    from xgboost import XGBClassifier
except Exception as e:
    raise RuntimeError("Please install xgboost: pip install xgboost") from e

# Utilities
import joblib
import json
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)



## 1) Load data
Place `creditcard.csv` in this folder before running.


In [ ]:

CSV_PATH = Path('creditcard.csv')
assert CSV_PATH.exists(), f"Could not find {CSV_PATH.resolve()}. Please place Kaggle's creditcard.csv next to this notebook."

df = pd.read_csv(CSV_PATH)
df.head()



## 2) Quick sanity checks


In [ ]:

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print(df['Class'].value_counts())
fraud_rate = df['Class'].mean()
print(f"Fraud rate: {fraud_rate:.4%}")



## 3) Preprocess
- Scale `Time` and `Amount` (the rest V1..V28 are already PCA-like).
- Train/valid/test split.


In [ ]:

# Features/target
X = df.drop(columns=['Class'])
y = df['Class']

# Scale Time and Amount
scaler = StandardScaler()
X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Save feature order for deployment
feature_cols = list(X.columns)
with open(ARTIFACT_DIR / 'feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)

# Save scaler
joblib.dump(scaler, ARTIFACT_DIR / 'scaler.joblib')

X_train.shape, X_test.shape



## 4) Handle Imbalance (for supervised model)
We'll use **RandomUnderSampler** for speed and simplicity.


In [ ]:

rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)
print("Before:", np.bincount(y_train))
print("After :", np.bincount(y_train_bal))



## 5) Unsupervised Anomaly Detection
We try **Isolation Forest** and **Local Outlier Factor** on the training set (unsupervised).  
We then evaluate on the test set.


In [ ]:

# Isolation Forest
iso = IsolationForest(
    n_estimators=200,
    contamination='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso.fit(X_train)

# Predict on test
iso_pred_raw = iso.predict(X_test)  # -1 = anomaly, 1 = normal
iso_pred = np.where(iso_pred_raw == -1, 1, 0)  # map anomaly->fraud(1)
cm_iso = confusion_matrix(y_test, iso_pred)
print("Isolation Forest Confusion Matrix:\n", cm_iso)

# LOF (fit_predict works on given data)
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination='auto',
    novelty=False,  # classic LOF (no .predict on new data); evaluate on test directly
    n_jobs=-1
)

# LOF needs to run on the test set for evaluation (not transductive on train+test together)
lof_pred_raw = lof.fit_predict(X_test)  # -1 anomaly, 1 normal
lof_pred = np.where(lof_pred_raw == -1, 1, 0)
cm_lof = confusion_matrix(y_test, lof_pred)
print("\nLocal Outlier Factor Confusion Matrix:\n", cm_lof)



## 6) Supervised Model (XGBoost)
Train an `XGBClassifier` on the **balanced** training set, evaluate on the test set.


In [ ]:

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist'
)

xgb.fit(X_train_bal, y_train_bal)
y_proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

cm_xgb = confusion_matrix(y_test, y_pred)
print("XGBoost Confusion Matrix:\n", cm_xgb)

print("\nClassification Report (XGBoost):\n")
print(classification_report(y_test, y_pred, digits=4))



## 7) ROC Curve


In [ ]:

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()



## 8) Save Artifacts for Deployment
- Trained XGBoost model
- Scaler
- Feature column order
- Confusion matrices


In [ ]:

# Save model
joblib.dump(xgb, ARTIFACT_DIR / 'model_xgb.joblib')

# Save confusion matrices
def plot_and_save_cm(cm, title, fname):
    import itertools
    plt.figure(figsize=(4,3))
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    classes = ['Legit(0)','Fraud(1)']
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / fname, bbox_inches='tight')
    plt.close()

plot_and_save_cm(cm_iso, 'Isolation Forest - Confusion Matrix', 'cm_iso.png')
plot_and_save_cm(cm_lof, 'Local Outlier Factor - Confusion Matrix', 'cm_lof.png')
plot_and_save_cm(cm_xgb, 'XGBoost - Confusion Matrix', 'cm_xgb.png')

# Save ROC data & AUC
with open(ARTIFACT_DIR / 'roc_metrics.json', 'w') as f:
    json.dump({'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': float(roc_auc)}, f)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
